This file is intended to download datasets if there is no existing backup.
The download speed is very low, and you can choose to download the datasets from other softwares or channels. 

# Download DAIC-WOZ Dataset

In [ ]:
import pandas as pd
import requests
from io import StringIO
from bs4 import BeautifulSoup
import re
from lxml import etree
import time 
import random
import os

In [ ]:
requests.packages.urllib3.disable_warnings() # to disable the warning


def create_soup(url):

    user_agent = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
                }
    resp = requests.get(url, headers=user_agent, verify=False)
    if resp.ok:
        return BeautifulSoup(resp.text,'html.parser')
    else:
        print('Error:',resp.status_code)
        return

In [ ]:
# Get the download links for the DAIC-WOZ dataset
DAIC_WOZ_page_url = r'https://dcapswoz.ict.usc.edu/wwwdaicwoz/'
soup = create_soup(DAIC_WOZ_page_url)
raw_data_url = soup.find_all('a',href=re.compile(r'_P.zip'))
ids = [id.get('href') for id in raw_data_url]
# https://dcapswoz.ict.usc.edu/wwwdaicwoz/311_P.zip
urls = [DAIC_WOZ_page_url + id for id in ids]


In [ ]:
# Download the DAIC-WOZ dataset
for url in urls:
    r = requests.get(url, stream=True)
    with open(f'.../data/raw/DAIC_WOZ/{url.split('/')[-1]}', 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    time.sleep(random.randint(1, 3)) # sleep for a while to avoid being blocked by the server
    print(f'{url.split("/")[-1]} downloaded')
print('All files downloaded')


Install with aiohttp.

In [ ]:
# !pip install nest_asyncio
import os
import aiohttp
import asyncio
import nest_asyncio

# 允许嵌套事件循环
nest_asyncio.apply()

async def download_file(session, url, file_path, semaphore):
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=60*60*60)) as response:  # 增加超时时间
                os.makedirs(os.path.dirname(file_path), exist_ok=True)
                with open(file_path, 'wb') as f:
                    while True:
                        chunk = await response.content.read(8192)
                        if not chunk:
                            break
                        f.write(chunk)
        except asyncio.TimeoutError:
            print(f"Timeout error for URL: {url}")

async def main(urls):
    semaphore = asyncio.Semaphore(max_concurrency)  # 设置最大并发数
    async with aiohttp.ClientSession() as session:
        tasks = []
        for url in urls:
            file_path = f'../../data/raw/DAIC_WOZ/{url.split("/")[-1]}'
            tasks.append(download_file(session, url, file_path, semaphore))
        await asyncio.gather(*tasks)

max_concurrency = 5  # 最大并发数

# 在 Jupyter Notebook 中运行异步函数
await main(urls)

# Download MELD dataset

In [ ]:
# Download the MELD dataset
MELD_url = 'https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz'
r = requests.get(MELD_url, stream=True)
with open(f'../../data/raw/MELD.Raw.tar.gz', 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)